# 60장 · 사용자별 장기 기억과 SQL 저장

장기 기억의 값은 대화가 끝난 뒤에도 다시 읽을 수 있습니다. 같은 설정 이름을 여러 사용자가 쓸 때 소유자도 조회 조건에 포함합니다.

alice의 응답 언어를 새 대화에서도 사용할 수 있도록 저장합니다. 같은 설정 이름을 쓰는 bob의 값과 섞이지 않는지 조회·수정·삭제 경로에서 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 59장의 결과 열기

같은 폴더에 내려받은 노트북을 여세요. 먼저 my-course/agent-59.json를 읽습니다. 파일이 없으면 시작 셀의 완성 예시로 연습할 수 있습니다.

In [ ]:
from pathlib import Path
import json
folder = Path.cwd()
if (folder.name == "daily" or (folder.name.startswith("chapter") and folder.name[7:].isdigit())):
    folder = folder.parent
work = folder / "my-course"
work.mkdir(exist_ok=True)
source_file = work / "agent-59.json"
if source_file.exists():
    previous = json.loads(source_file.read_text(encoding="utf-8"))
else:
    previous = {'record': {'owner': 'alice', 'id': 'loan-01', 'text': '대출 기간은 14일입니다.'},
     'owner': 'alice',
     'turns': [{'role': 'assistant', 'content': '대출 기간은 14일입니다.'},
               {'role': 'user', 'content': '앞으로는 짧게 답해 주세요.'}]}
print("내 파일" if source_file.exists() else "완성 예시", previous["record"]["text"])

완성 예시 대출 기간은 14일입니다.


## 사용자 ID와 설정 키의 복합 조건

메시지는 어느 대화에서 나온 말인지가 중요합니다. 사용자 선호는 새 대화에서도 찾기 위해 사용자와 설정 이름을 함께 키로 삼습니다. 아래는 저장소 조회 범위를 메모리로 재현한 예입니다. PostgreSQL의 영속성은 이 셀에서 측정하지 않습니다.


### 셀 01 · 사용자별 설정 키 조회

새 대화 ID 없이도 해당 사용자의 선호를 찾는지 보세요.

설정 키에 사용자 ID는 들어 있지만 대화 ID는 없습니다. 같은 사용자가 대화 A와 B에서 설정을 읽어도 같은 키를 조회합니다. 이 셀이 보여 주는 것은 키의 구성이고, 새 세션 연결은 따로 확인해야 합니다.


In [1]:
memory = {("alice", "language"): "한국어"}
owner = "alice"  # 앱이 확인해 전달한 수업용 사용자
print(memory.get((owner, "language")))

한국어


('alice', 'language')를 조회해 ‘한국어’를 얻습니다. 이 커널이 같은 딕셔너리를 사용하는 동안에는 대화 ID가 바뀌어도 같은 키를 쓸 수 있습니다.


### 대화를 넘어 사용할 설정

<p>language 같은 설정을 다시 쓰려면 누구의 선호인지 함께 저장해야 합니다. 이 셀에서 조회의 기준은 두 값으로 만든 키 ("alice", "language")입니다.</p>


### 셀 02 · 동일한 키의 소유자 분리

설정 이름은 그대로 두고 사용자를 바꿉니다.


In [2]:
print(memory.get(("bob", "language")))

None


None은 bob의 항목이 없다는 뜻입니다. 여기서 alice의 값으로 대체하면 사용자 경계가 무너집니다. DB 연결이 끊긴 상황은 조회 성공 후의 빈 값과 따로 처리합니다.

다른 사용자에게 같은 설정 이름으로 조회했을 때 None이 반환됩니다. 설정 이름만 키로 사용하면 먼저 저장한 언어가 다른 사용자에게도 적용될 수 있습니다. 사용자 식별자까지 키에 포함해야 값의 소유 범위가 보존됩니다.


#### 같은 선호 키를 여러 사람이 쓸 때

<mark>조회 조건에는 설정 이름과 사용자 ID가 모두 필요합니다.</mark> 두 사용자가 각각 언어를 저장한 뒤 한 사람의 값만 지워 보세요. 다른 사용자의 선호가 남아야 개인 설정의 수정 범위가 맞습니다.

선수 개념 · 54장 SQL 매개변수 바인딩과 트랜잭션

### 같은 설정 이름과 다른 소유자

<p>alice의 language가 저장되어 있어도 ("bob", "language")는 다른 키입니다. 빈 조회의 None을 다른 사용자의 값으로 채우면 소유자 분리가 깨집니다.</p>


## 선호 정보의 수정과 삭제

설정 이름 language는 두 사용자가 함께 씁니다. 사용자가 보낸 문장에서 이름을 추측하면 같은 설정을 다른 사람의 값으로 덮을 수 있습니다. 이 실습의 사용자 ID는 앱에서 확인해 받았다고 가정합니다.


### 과제 03 · 특정 사용자의 설정 갱신

alice의 값은 유지하고 bob의 언어를 영어로 저장한 뒤 bob의 값을 읽으세요.


In [3]:
# (사용자, 설정 이름) 키로 bob의 값을 저장하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>memory[(&#x27;bob&#x27;, &#x27;language&#x27;)] = &#x27;영어&#x27;
print(memory[(&#x27;bob&#x27;, &#x27;language&#x27;)])</code></pre>
<p>실행 결과</p><pre><code>영어</code></pre>
<p>영어는 bob 행에만 들어갑니다. DB에서는 사용자와 키가 모두 일치하는 행을 수정해야 같은 결과를 얻습니다.</p>
</details>


### 한 사용자 설정만 수정하기

<p>딕셔너리에서 ("bob", "language")에 값을 대입하면 그 키의 항목만 생기거나 바뀝니다. 삭제할 때도 (사용자, 설정 이름) 두 값을 모두 지정합니다.</p>


### 과제 04 · 소유자별 설정 삭제

샘플 복사본에서 alice의 language만 지우고 남은 bob의 설정을 출력하세요.


In [4]:
saved = {('alice', 'language'): '한국어', ('bob', 'language'): '영어'}
# alice의 항목만 지우세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>saved = {(&#x27;alice&#x27;, &#x27;language&#x27;): &#x27;한국어&#x27;, (&#x27;bob&#x27;, &#x27;language&#x27;): &#x27;영어&#x27;}
del saved[(&#x27;alice&#x27;, &#x27;language&#x27;)]
print(saved)</code></pre>
<p>실행 결과</p><pre><code>{(&#x27;bob&#x27;, &#x27;language&#x27;): &#x27;영어&#x27;}</code></pre>
<p>bob의 설정이 남습니다. 저장한 대화 전체를 지운 작업과 별개입니다. 실제 DB에서는 새 연결에서 같은 조회를 반복해 commit이 유지됐는지도 확인합니다.</p>
</details>


## SQL WHERE로 소유자별 조회

제공 PostgreSQL 저장소는 SELECT value FROM course_memories WHERE owner=%s AND memory_key=%s에 (user_id, memory_key)를 따로 전달합니다. 아래는 메모리 SQLite DB에 같은 행을 넣어 조건을 직접 실행하는 실습입니다. 두 드라이버가 쓰는 자리표시자가 다릅니다.


### 셀 05 · SQL 매개변수로 범위 제한

SQLite의 ? 두 자리에 소유자와 설정 이름을 순서대로 전달합니다.


In [5]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE course_memories(owner TEXT, memory_key TEXT, value TEXT, PRIMARY KEY(owner,memory_key))")
db.executemany("INSERT INTO course_memories VALUES (?,?,?)",
               [("alice", "language", "한국어"), ("bob", "language", "영어")])
row = db.execute("SELECT value FROM course_memories WHERE owner=? AND memory_key=?",
                 ("alice", "language")).fetchone()
print(row[0])

한국어


한국어는 두 조건을 모두 만족한 행에서 읽었습니다. PostgreSQL 확인 환경에서는 같은 두 사용자를 준비한 뒤 새 연결에서 조회하고 앱을 재시작해 다시 읽습니다. 이 셀의 메모리 DB는 연결을 닫으면 사라집니다.


#### 두 사용자의 language 중 하나만 조회하기

```mermaid
flowchart TD
ntable["기억 테이블"]
  nfilter["WHERE 조건"]
  nalice["alice 행"]
  nbob["bob 행"]
  nvalue["fetchone"]
  ntable --> nfilter
  nfilter --> nalice
  nfilter --> nbob
  nalice --> nvalue
  nbob --> nvalue
```

SQLite가 두 조건을 동시에 만족하는 행에서 value를 가져옵니다.

### 복합 기본 키와 조회 조건

<p>PRIMARY KEY(owner, memory_key)는 같은 사용자의 같은 설정이 두 행이 되는 것을 막습니다. SELECT의 WHERE에도 같은 두 열을 써서 같은 이름의 남의 설정을 제외합니다.</p>


## 복습과 실습 점검

### 과제 06 · 한 사용자의 설정만 SQL로 지우기

같은 language 키를 가진 행 두 개 중 alice의 행만 지우고 남은 소유자와 값을 확인하세요. 다시 실행할 수 있도록 SAVEPOINT를 만든 뒤 삭제하고, 조회가 끝나면 그 지점으로 돌아가세요.


In [ ]:
db.execute("SAVEPOINT check_delete")
# 삭제 후 남은 행을 확인하세요.
db.execute("ROLLBACK TO check_delete")
db.execute("RELEASE check_delete")

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>db.execute(&quot;SAVEPOINT check_delete&quot;)
db.execute(&quot;DELETE FROM course_memories WHERE owner=? AND memory_key=?&quot;, (&quot;alice&quot;, &quot;language&quot;))
print(db.execute(&quot;SELECT owner, value FROM course_memories ORDER BY owner&quot;).fetchall())
db.execute(&quot;ROLLBACK TO check_delete&quot;)
db.execute(&quot;RELEASE check_delete&quot;)</code></pre>
<p>실행 결과</p><pre><code>[('bob', '영어')]</code></pre>
<p>삭제와 조회에 같은 연결을 써서 변경 직후의 상태를 확인했습니다. DELETE의 조건을 memory_key 하나로 줄이면 bob의 행까지 사라집니다. SAVEPOINT로 돌아가면 이 과제에서 수행한 삭제만 취소됩니다.</p>
</details>


## 새 대화에서 같은 사람의 선호만 불러오기

대화방을 바꿔도 답변 언어는 유지돼야 합니다. 59장의 thread_id 대신 서비스가 확인한 사용자 ID로 선호를 찾습니다. 저장 함수가 다른 사람의 같은 이름 설정에 영향을 주는지도 살펴봅니다.

### 셀 08 · 사용자별로 같은 language 키 보관하기

LangGraph의 InMemoryStore에 alice와 bob의 기본 언어를 따로 저장하세요. 아래 함수는 어느 대화방에서 불러도 사용자와 기억 종류로 같은 공간을 찾습니다.

In [6]:
from langgraph.store.memory import InMemoryStore

profile_store = InMemoryStore()
def preference_space(user_id):
    return ("users", user_id, "preferences")

profile_store.put(preference_space("alice"), "language", {"value": "한국어"})
profile_store.put(preference_space("bob"), "language", {"value": "영어"})
def preferred_language(user_id):
    item = profile_store.get(preference_space(user_id), "language")
    return item.value["value"] if item else None

print(preferred_language("alice"), preferred_language("bob"))

한국어 영어


두 키의 이름이 language로 같아도 결과는 한국어와 영어입니다. user_id가 다른 네임스페이스를 읽었기 때문입니다. 이 저장소는 같은 Python 프로세스에서 대화 사이에 재사용할 수 있고, 커널을 끄면 사라집니다.

### 과제 09 · 기억을 고친 뒤 한 사람의 설정만 삭제하기

alice의 언어를 일본어로 수정하고 다시 읽으세요. alice가 이 설정을 지워 달라고 하면 해당 네임스페이스의 language만 삭제하세요. 삭제 뒤 alice와 bob의 값을 각각 확인하세요.

In [7]:
# put으로 한 사람의 값을 바꾸고, delete에 같은 네임스페이스와 키를 넘기세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>profile_store.put(preference_space(&quot;alice&quot;), &quot;language&quot;, {&quot;value&quot;: &quot;일본어&quot;})
print(preferred_language(&quot;alice&quot;))
profile_store.delete(preference_space(&quot;alice&quot;), &quot;language&quot;)
print(preferred_language(&quot;alice&quot;), preferred_language(&quot;bob&quot;))</code></pre>
<p>실행 결과</p><pre><code>일본어
None 영어</code></pre>
<p>첫 줄은 일본어이고 마지막 줄은 None과 영어입니다. alice의 새 값과 삭제가 모두 같은 키를 대상으로 했고 bob의 설정은 유지됐습니다. 네임스페이스에 넣을 사용자 ID는 로그인 결과 등 서비스가 신뢰하는 값으로 정합니다.</p>
</details>

### 셀 10 · 선택 실습 · PostgreSQL의 복합 키에 같은 갱신 적용하기

53장의 PostgreSQL 접속 환경과 psycopg가 준비된 수업에서 실행할 수 있습니다. 임시 테이블에서 사용자와 설정명이 같은 데이터를 다시 저장하고, 행이 몇 개 남았는지 읽습니다. 기본 설정에서는 접속하지 않습니다.

In [8]:
run_profile_db = False
if run_profile_db:
    import psycopg
    with psycopg.connect(connect_timeout=3) as conn:
        conn.execute("CREATE TEMP TABLE pref_demo (owner text, key text, val text, PRIMARY KEY(owner, key))")
        upsert = """INSERT INTO pref_demo VALUES (%s, %s, %s)
                    ON CONFLICT(owner, key) DO UPDATE SET val = EXCLUDED.val"""
        conn.execute(upsert, ("alice", "language", "한국어"))
        conn.execute(upsert, ("alice", "language", "일본어"))
        print(conn.execute("SELECT owner, key, val FROM pref_demo").fetchall())
else:
    print("PostgreSQL 수업 환경에서 연결을 선택하세요.")

PostgreSQL 수업 환경에서 연결을 선택하세요.


연결한 경우 마지막 조회에서 alice의 language 한 행을 확인하세요. 두 번째 입력의 값이 일본어로 반영되어야 합니다. 이 셀은 임시 테이블의 행 갱신을 확인하는 용도입니다. 서비스 재시작 뒤 장기 보존을 시험하려면 서비스의 영속 테이블에서 별도로 확인해야 합니다.

<mark>사용자와 설정 이름을 함께 정해야 같은 사람의 같은 기억을 갱신할 수 있습니다.</mark> 대화방 ID로만 저장하면 새 대화에서 이전 언어 설정을 찾기 어렵습니다. 사용자별 공간에 공통 키 language를 두면 두 요청이 같은 설정을 가리킵니다.

통합 서비스에서 선택된 언어가 어떤 사용자에게서 왔는지 확인할 수 있습니다.

## SQLite UPSERT로 사용자별 답변 선호 보관하기

59장의 마지막 요청에서 고객이 원하는 답변 길이를 고릅니다. 대화가 끝나도 같은 사용자에게만 이 선호가 적용되도록 파일 데이터베이스에 저장하세요.

In [ ]:
import sqlite3
record = previous["record"]
owner = previous["owner"]
last = previous["turns"][-1]["content"]
style = "short" if "짧게" in last else "standard"
profile_path = work / "preferences.sqlite"
with sqlite3.connect(profile_path) as profiles:
    profiles.execute("CREATE TABLE IF NOT EXISTS prefs (owner TEXT PRIMARY KEY, style TEXT)")
    profiles.execute("INSERT INTO prefs VALUES (?, ?) ON CONFLICT(owner) DO UPDATE SET style=excluded.style",
                     (owner, style))
with sqlite3.connect(profile_path) as reopened:
    saved_style = reopened.execute("SELECT style FROM prefs WHERE owner=?", (owner,)).fetchone()[0]
print(owner, saved_style)

alice short


새 연결에서 다시 읽어도 기본 자료의 alice에게 short가 남습니다. owner가 기본 키여서 같은 사용자의 선호를 두 번 저장해도 그 사용자의 행은 한 개입니다.

### 직접 확인하기

저장한 사용자만 대상으로 행 수를 세세요. 이 셀을 다시 실행한 다음에도 같은 소유자의 행이 한 건이어야 합니다.

In [ ]:
# profiles를 다시 열어 owner에 해당하는 행 수를 세세요.

<details><summary>정답 코드와 실행 결과 보기</summary><pre><code>with sqlite3.connect(profile_path) as connection:
    total = connection.execute(&quot;SELECT COUNT(*) FROM prefs WHERE owner=?&quot;, (owner,)).fetchone()[0]
print(total)</code></pre><p>기본 자료로 실행한 결과</p><pre><code>1</code></pre><p>PRIMARY KEY(owner)와 ON CONFLICT가 같은 소유자의 선호를 새 값으로 바꿉니다.</p></details>

### 61장에 넘길 결과 저장하기

61장은 저장한 선호와 문서 원문을 읽는 도구를 MCP로 공개합니다. 저장 셀을 다시 실행하면 같은 파일에 이번 결과를 기록합니다.

In [ ]:
product = {"record": record, "style": saved_style}
target = work / "agent-60.json"
target.write_text(json.dumps(product, ensure_ascii=False, indent=2), encoding="utf-8")
print(target.name)

agent-60.json
